
**Created by:**

__Viktor Varga__

<br>
<img src="../_files/nb_im/ik_logo.png" style="display:inline-block" width='40%'>
<hr>

# Python tutorial - Iterators and generators


## Iterators

One way to support lazy evaluation is by using iterators. In older versions of Python (Python 2), many built-in operations returned one or more lists. This is not ideal when you need to generate large lists but only continue working with the first few elements. In Python 3, most of these built-in operations return iterators. An iterator produces the next element only when we request it.


In Python 3, an iterable type can be implemented by a class that has an `__iter__()` member function. Iterable types include standard, built-in container types (list, tuple, set, dictionary, etc.). The `__iter__()` member function of these types must return an iterator that iterates over the elements of the container. The iterator type can be implemented by a class that has a `__next__()` member function. This function is responsible for returning the next element, so the iterator must know its current position inside the iterable object.

It is possible for a single class to be both iterable and an iterator. Often, these are such types that do not store elements physically, but only generate the next element on demand. For instances of such classes, the `__iter__()` member function usually returns `self`, the current instance.

Below we implement a new iterator that yields sliding-window-style intervals over integers.


In [ ]:
# custom iterator class: returning sliding windows

class SlidingWindowIterator:

  def __init__(self, n, window_len):
    self.n = n
    self.window_len = window_len
    self.current = 0

  def __iter__(self):
    return self

  def __next__(self):
    if self.current+self.window_len-1 < self.n:
      old_current = self.current
      self.current += 1
      return list(range(old_current, old_current+self.window_len))
    else:
      raise StopIteration()


Let's instantiate our new iterator class!


In [ ]:
print("The sliding windows:")
it = SlidingWindowIterator(5,3)  # windows with a length of 3 from 0 to 5
for item in it:
  print("    ", item)

The sliding windows:
     [0, 1, 2]
     [1, 2, 3]
     [2, 3, 4]


In [ ]:
print("Let's try to continue the iteration!")
for item in it:
  print("    ", item)

Let's try to continue the iteration!


As we can see, the first `for` loop iterated through the iterator instance and exhausted it. Because of this, the next `for` loop cannot read anything from it. A `for` loop repeatedly calls the iterator's `__next__()` method until it raises a `StopIteration` exception.

Let's instantiate the iterator class again and step through it by calling `__next__()` manually. The built-in `next()` function does the same: it calls the `__next__()` method of the iterator passed as its argument.


In [ ]:
# new iterator
it = SlidingWindowIterator(5,3)
try:
  print("The first element: ", it.__next__())
  print("The second element: ", it.__next__())
  print("The third element: ", next(it))   # Python builtin next() function calls __next__()
  print("The fourth element: ", next(it))
except StopIteration:
  print("StopIteration was raised: reached end of iterator.")


The first element:  [0, 1, 2]
The second element:  [1, 2, 3]
The third element:  [2, 3, 4]
StopIteration was raised: reached end of iterator.


Many built-in Python 3 functions return objects that rely on iterators. One example is the `range` class returned by the built-in `range()` function. The `range` object itself is not an iterator, but it implements `__iter__()`, which returns an iterator that can be iterated over.


In [ ]:
range_obj = range(5, 10, 2)
print("The type of range function's return value is ", type(range_obj))
print("Iterating through the range object's iterator:")
for item in range_obj:
  print("    ", item)

# Calling next(range_obj) would result in: "TypeError: 'range' object is not an iterator"

# Instead, get the range object's iterator
range_obj_it = range_obj.__iter__()
print("\nThe type of range object's iterator ", type(range_obj_it))

print("Now we can call iterate through the iterator by calling next() multiple times:")
try:
  print("    The first element: ", range_obj_it.__next__())
  print("    The second element: ", range_obj_it.__next__())
  print("    The third element: ", next(range_obj_it)) # Python builtin next() function calls __next__()
  print("    The fourth element: ", next(range_obj_it))
except StopIteration:
  print("    StopIteration was raised: reached end of iterator.")


The type of range function's return value is  <class 'range'>
Iterating through the range object's iterator:
     5
     7
     9

The type of range object's iterator  <class 'range_iterator'>
Now we can call iterate through the iterator by calling next() multiple times:
    The first element:  5
    The second element:  7
    The third element:  9
    StopIteration was raised: reached end of iterator.


That is, when iterating in a `for` loop, Python first calls the `__iter__()` method of the object after the `in` keyword. If the object is already an iterator, its `__iter__()` method returns `self`, i.e. itself. A `range` object is not an iterator, but calling `__iter__()` on it returns a `range_iterator` object, which is an iterator.

Why is this useful? Below you can see the difference:


In [ ]:
range_obj = range(3)
range_obj_it = range(3).__iter__()

try:
    print("Starting iteration#1...")
    for item in range_obj:
        print("    ", item)
    print("Starting iteration#2...")
    for item in range_obj:
        print("    ", item)
except StopIteration:
    print("    StopIteration was raised: reached end of iterator.")

print("\nA range object can be used for multiple iterations. Now let's try the iterator itself:")

try:
    print("Starting iteration#1...")
    for item in range_obj_it:
        print("    ", item)
    print("Starting iteration#2...")
    for item in range_obj_it:
        print("    ", item)
except StopIteration:
    print("    StopIteration was raised: reached end of iterator.")

print("\nThe iterator itself expires after it is iterated through.")

Starting iteration#1...
     0
     1
     2
Starting iteration#2...
     0
     1
     2

A range object can be used for multiple iterations. Now let's try the iterator itself:
Starting iteration#1...
     0
     1
     2
Starting iteration#2...

The iterator itself expires after it is iterated through.


So a `range` object can be reused multiple times, while the iterator returned by it can be used only once.

You get a similar kind of object if you query the key set of a `dict` data structure:


In [ ]:
my_dict = {1:"egy", 2:"ketto", 3:"harom"}

my_keys = my_dict.keys()
print("The key set: ", my_keys)
print("The type of the key set: ", type(my_keys))
print("The type of the key set's iterator: ", type(my_keys.__iter__()))

print("Starting iteration#1...")
for item in my_keys:
    print("    ", item)
print("Starting iteration#2...")
for item in my_keys:
    print("    ", item)

The key set:  dict_keys([1, 2, 3])
The type of the key set:  <class 'dict_keys'>
The type of the key set's iterator:  <class 'dict_keyiterator'>
Starting iteration#1...
     1
     2
     3
Starting iteration#2...
     1
     2
     3


## Generators

A shorter way to implement iterators. A generator's `yield` statement is similar to `return`, but after yielding a value it suspends execution and continues at the next iteration step, until it encounters another `yield`. While suspended, it preserves the values of its local variables. A generator finishes only when it reaches a `return` statement or the end of its code.

Like iterators, generators can be advanced step by step; the `generator` object implements both `__iter__()` and `__next__()`.

Now, let's implement the iterator class defined at the beginning of the notebook, `SlidingWindowIterator`, as a generator!


In [ ]:
# generator: returning sliding windows

def sliding_window_generator(n, window_len):
  for idx in range(0,n-window_len+1):
    print("    Upcoming yield!")   # this is only here for demonstration purposes
    yield list(range(idx, idx+window_len))

it = sliding_window_generator(5,3)
print("The type of the generator: ", type(it))
print("The generator returns self on __iter__() call: ", type(it.__iter__()))
print("  ... in short, it is an iterator.")

print("Now let's iterate through it: ")
for item in it:
    print("    ", item)

print("\nLet's try to do one more iteration step on it:")
try:
    next(it)
except StopIteration:
    print("    StopIteration was raised: reached end of iterator.")

print("\nAs we see, generators are like iterators, when they reach the end, they expire...")

it = sliding_window_generator(5,3)
print("But we can instantiate a new one: ", next(it), " ...")


The type of the generator:  <class 'generator'>
The generator returns self on __iter__() call:  <class 'generator'>
  ... in short, it is an iterator.
Now let's iterate through it: 
    Upcoming yield!
     [0, 1, 2]
    Upcoming yield!
     [1, 2, 3]
    Upcoming yield!
     [2, 3, 4]

Let's try to do one more iteration step on it:
    StopIteration was raised: reached end of iterator.

As we see, generators are like iterators, when they reach the end, they expire...
    Upcoming yield!
But we can instantiate a new one:  [0, 1, 2]  ...


### Generator expression

Similarly to list/set/dict comprehensions, a generator can often be written in a shorter form as a generator expression. If you exhaust the iterator produced by a generator expression, you must define it again. The `tee()` function from the `itertools` package can be used to copy iterator objects, with care (see the documentation).


In [ ]:
window_len = 3
n = 5

sliding_window_generator = (list(range(i, i+window_len)) for i in range(n-window_len+1))

print("\nLet's iterate through it:")
for item in sliding_window_generator:
  print("    ", item)

# to reuse the generator expression, we must define it again


Let's iterate through it:
     [0, 1, 2]
     [1, 2, 3]
     [2, 3, 4]
